# RetailMind AI ? Customer Churn
## Days 1?4

This notebook contains the code to prepare data, train models, and save results. Run cells from top to bottom. Each day uses the same held-out test customers.

## Before starting

Put the Telco Customer Churn CSV in `data/`. The next cell loads it and creates a training group and a test group.

In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.feature_selection import SelectKBest, f_classif

# Find the project folder from either the root or notebooks/.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks": PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports"
MODEL_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

# Read and prepare the Telco customer CSV.
file_names = ["Telco-Customer-Churn.csv", "WA_Fn-UseC_-Telco-Customer-Churn.csv", "telco_customer_churn.csv"]
csv_path = next((DATA_DIR / name for name in file_names if (DATA_DIR / name).is_file()), None)
if csv_path is None: raise FileNotFoundError("Put the Telco churn CSV in the data folder.")
data = pd.read_csv(csv_path)
data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")
y = data.pop("Churn").map({"No": 0, "Yes": 1})
if y.isna().any(): raise ValueError("Churn should contain only Yes and No.")
X = data.drop(columns=["customerID"], errors="ignore")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Prepare numbers and text inside each model pipeline.
def make_preprocessor(frame):
    number_columns = frame.select_dtypes(include="number").columns.tolist()
    text_columns = frame.select_dtypes(exclude="number").columns.tolist()
    return ColumnTransformer([
        ("numbers", Pipeline([("fill", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), number_columns),
        ("text", Pipeline([("fill", SimpleImputer(strategy="most_frequent")), ("encode", OneHotEncoder(handle_unknown="ignore"))]), text_columns),
    ])
def make_model(estimator):
    return Pipeline([("prepare", make_preprocessor(X_train)), ("model", estimator)])
def score_model(name, model):
    guess = model.predict(X_test)
    probability = model.predict_proba(X_test)[:, 1]
    result = {"Model": name, "Accuracy": accuracy_score(y_test, guess), "Precision": precision_score(y_test, guess, zero_division=0), "Recall": recall_score(y_test, guess, zero_division=0), "F1": f1_score(y_test, guess, zero_division=0), "ROC_AUC": roc_auc_score(y_test, probability)}
    print(pd.Series(result).to_string())
    print("Confusion matrix [correct no, false alarm; missed churn, found churn]:")
    print(confusion_matrix(y_test, guess))
    return result
print("Training rows:", len(X_train), " Test rows:", len(X_test))

## Day 1 ? A simple first model

Logistic Regression is the baseline. The pipeline fills missing values, scales numbers, and converts text answers into numbers.

In [ ]:
day1_model = make_model(LogisticRegression(max_iter=1500, class_weight="balanced", random_state=42))
day1_model.fit(X_train, y_train)
day1_result = score_model("Day 1 Logistic Regression", day1_model)
joblib.dump(day1_model, MODEL_DIR / "churn_day1_notebook.joblib")
pd.DataFrame([day1_result]).to_csv(REPORT_DIR / "day1_notebook_metrics.csv", index=False)

## Day 2 ? Compare models

Cross-validation tries each model on different parts of training data. We choose the model with the highest average recall for customers who leave.

In [ ]:
candidates = {
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=250, class_weight="balanced", random_state=42, n_jobs=-1),
    "SVM": SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=42),
}
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
models = {}
rows = []
for name, estimator in candidates.items():
    model = make_model(estimator)
    model.fit(X_train, y_train)
    models[name] = model
    row = score_model("Day 2 " + name, model)
    row["CV_Recall"] = cross_val_score(make_model(estimator), X_train, y_train, scoring="recall", cv=cv).mean()
    rows.append(row)
day2_results = pd.DataFrame(rows).sort_values("CV_Recall", ascending=False)
day2_name = day2_results.iloc[0]["Model"].replace("Day 2 ", "")
day2_model = models[day2_name]
display(day2_results)
joblib.dump(day2_model, MODEL_DIR / "churn_day2_notebook.joblib")
day2_results.to_csv(REPORT_DIR / "day2_notebook_comparison.csv", index=False)

## Day 3 ? Tune model settings

Grid search tries a few settings using training data only. It picks the best average recall, then reports its result on the test customers.

In [ ]:
searches = {
    "Decision Tree": (DecisionTreeClassifier(class_weight="balanced", random_state=42), {"model__max_depth": [4, 8, None]}),
    "Random Forest": (RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1), {"model__n_estimators": [150, 250], "model__max_depth": [None, 10]}),
    "SVM": (SVC(kernel="rbf", probability=True, class_weight="balanced"), {"model__C": [0.5, 1, 3]}),
}
search_results = []
best_models = {}
for name, (estimator, settings) in searches.items():
    search = GridSearchCV(make_model(estimator), settings, scoring="recall", cv=cv, n_jobs=1)
    search.fit(X_train, y_train)
    best_models[name] = search.best_estimator_
    search_results.append({"Model": name, "CV_Recall": search.best_score_, "Best_Settings": search.best_params_})
day3_results = pd.DataFrame(search_results).sort_values("CV_Recall", ascending=False)
day3_name = day3_results.iloc[0]["Model"]
day3_model = best_models[day3_name]
day3_score = score_model("Day 3 Tuned " + day3_name, day3_model)
display(day3_results)
joblib.dump(day3_model, MODEL_DIR / "churn_day3_notebook.joblib")
day3_results.to_csv(REPORT_DIR / "day3_notebook_tuning.csv", index=False)
pd.DataFrame([day3_score]).to_csv(REPORT_DIR / "day3_notebook_metrics.csv", index=False)

## Day 4 ? Add useful customer details

Create a few understandable features, such as tenure group and average monthly charge. Then train a Random Forest with feature selection and compare it with Day 3.

In [ ]:
def add_customer_features(frame):
    frame = frame.copy()
    tenure = pd.to_numeric(frame["tenure"], errors="coerce")
    frame["tenure_group"] = pd.cut(tenure, [-1, 12, 24, 48, 72], labels=["0-12", "13-24", "25-48", "49-72"]).astype("object")
    total = pd.to_numeric(frame["TotalCharges"], errors="coerce")
    frame["average_charge_per_month"] = total / tenure.replace(0, np.nan)
    frame["is_month_to_month"] = (frame["Contract"] == "Month-to-month").astype(int)
    frame["uses_electronic_check"] = (frame["PaymentMethod"] == "Electronic check").astype(int)
    return frame

# These features are simple row-by-row calculations, so they can be made before the pipeline.
X_train_day4 = add_customer_features(X_train)
X_test_day4 = add_customer_features(X_test)
day4_model = Pipeline([
    ("prepare", make_preprocessor(X_train_day4)),
    ("select", SelectKBest(f_classif, k=30)),
    ("model", RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight="balanced", random_state=42, n_jobs=-1)),
])
day4_model.fit(X_train_day4, y_train)
day4_guess = day4_model.predict(X_test_day4)
day4_probability = day4_model.predict_proba(X_test_day4)[:, 1]
day4_result = {"Model": "Day 4 Random Forest", "Accuracy": accuracy_score(y_test, day4_guess), "Precision": precision_score(y_test, day4_guess, zero_division=0), "Recall": recall_score(y_test, day4_guess, zero_division=0), "F1": f1_score(y_test, day4_guess, zero_division=0), "ROC_AUC": roc_auc_score(y_test, day4_probability)}
print(pd.Series(day4_result).to_string())
print("Confusion matrix:")
print(confusion_matrix(y_test, day4_guess))
day4_comparison = pd.DataFrame([score_model("Day 3 " + day3_name, day3_model), day4_result])
display(day4_comparison)
joblib.dump(day4_model, MODEL_DIR / "churn_day4_notebook.joblib")
day4_comparison.to_csv(REPORT_DIR / "day4_notebook_comparison.csv", index=False)

## Saved files

This notebook saves its models and result tables with notebook-specific filenames in `models/` and `reports/`.